# Inspect ResNet293 Model

This notebook loads the WeSpeaker ResNet293 PyTorch checkpoint, runs one audio sample through it, and shows how tensor shapes change across the main layers.


In [ ]:
import sys
import types
from pathlib import Path
import importlib.util

import pandas as pd
import soundfile as sf
import torch
import torchaudio
import torchaudio.compliance.kaldi as kaldi
import yaml


In [ ]:
# ===== Config =====
REPO_ROOT = Path('/home/SpeakerRec/BioVoice')
MODEL_DIR = REPO_ROOT / 'data' / 'models' / 'wespeaker_resnet293_lm'
AUDIO_PATH = REPO_ROOT / 'data' / 'wavs' / 'eden_001.wav'
WESPEAKER_MODELS_DIR = REPO_ROOT / 'external' / 'wespeaker' / 'wespeaker' / 'models'

CONFIG_PATH = MODEL_DIR / 'config.yaml'
CKPT_PATH = MODEL_DIR / 'avg_model.pt'
TARGET_SR = 16000

print('CONFIG EXISTS =', CONFIG_PATH.exists())
print('CKPT EXISTS =', CKPT_PATH.exists())
print('AUDIO EXISTS =', AUDIO_PATH.exists())
print('WESPEAKER_MODELS_DIR EXISTS =', WESPEAKER_MODELS_DIR.exists())


In [ ]:
# Load only the exact WeSpeaker model files we need.
wespeaker_pkg = types.ModuleType('wespeaker')
wespeaker_models_pkg = types.ModuleType('wespeaker.models')
sys.modules['wespeaker'] = wespeaker_pkg
sys.modules['wespeaker.models'] = wespeaker_models_pkg

pool_spec = importlib.util.spec_from_file_location(
    'wespeaker.models.pooling_layers',
    str(WESPEAKER_MODELS_DIR / 'pooling_layers.py'),
)
pool_mod = importlib.util.module_from_spec(pool_spec)
sys.modules['wespeaker.models.pooling_layers'] = pool_mod
pool_spec.loader.exec_module(pool_mod)

resnet_spec = importlib.util.spec_from_file_location(
    'wespeaker.models.resnet',
    str(WESPEAKER_MODELS_DIR / 'resnet.py'),
)
resnet_mod = importlib.util.module_from_spec(resnet_spec)
sys.modules['wespeaker.models.resnet'] = resnet_mod
resnet_spec.loader.exec_module(resnet_mod)

ResNet293 = resnet_mod.ResNet293
print('ResNet293 constructor loaded.')


In [ ]:
with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    configs = yaml.safe_load(f)

print('model =', configs['model'])
print('model_args =', configs['model_args'])
print('fbank_args =', configs['dataset_args']['fbank_args'])

model = ResNet293(**configs['model_args'])
state = torch.load(CKPT_PATH, map_location='cpu')

if isinstance(state, dict):
    if 'state_dict' in state:
        state = state['state_dict']
    elif 'model' in state:
        state = state['model']

if isinstance(state, dict):
    state = { (k[7:] if k.startswith('module.') else k): v for k, v in state.items() }

missing, unexpected = model.load_state_dict(state, strict=False)
model.eval()

print('Missing keys:', missing)
print('Unexpected keys:', unexpected)
model


In [ ]:
waveform_np, sr = sf.read(str(AUDIO_PATH))
print('loaded waveform raw shape =', waveform_np.shape, '| sr =', sr)

if waveform_np.ndim == 1:
    waveform_np = waveform_np[None, :]
else:
    waveform_np = waveform_np.T

waveform = torch.from_numpy(waveform_np).float()
if sr != TARGET_SR:
    waveform = torchaudio.functional.resample(waveform, sr, TARGET_SR)
    sr = TARGET_SR

waveform = waveform[:1, :]
waveform = waveform * (1 << 15)

fbank_args = configs['dataset_args']['fbank_args']
feats = kaldi.fbank(
    waveform,
    num_mel_bins=fbank_args['num_mel_bins'],
    frame_length=fbank_args['frame_length'],
    frame_shift=fbank_args['frame_shift'],
    dither=0.0,
    sample_frequency=TARGET_SR,
)

feats = feats.unsqueeze(0)
print('input feature tensor shape =', tuple(feats.shape))
display(feats[0, :5, :8])


In [ ]:
shape_rows = []
hooks = []

def _shape_of(x):
    if isinstance(x, torch.Tensor):
        return tuple(x.shape)
    if isinstance(x, (list, tuple)):
        return [_shape_of(v) for v in x]
    return str(type(x))

def register_shape_hook(module_name, module):
    def hook(_module, inputs, outputs):
        shape_rows.append({
            'layer': module_name,
            'input_shape': _shape_of(inputs[0] if len(inputs) == 1 else inputs),
            'output_shape': _shape_of(outputs),
        })
    hooks.append(module.register_forward_hook(hook))

modules_to_trace = [
    ('conv1', model.conv1),
    ('bn1', model.bn1),
    ('layer1', model.layer1),
    ('layer2', model.layer2),
    ('layer3', model.layer3),
    ('layer4', model.layer4),
    ('pool', model.pool),
    ('seg_1', model.seg_1),
]

for name, module in modules_to_trace:
    register_shape_hook(name, module)

with torch.no_grad():
    outputs = model(feats)
    emb = outputs[-1] if isinstance(outputs, tuple) else outputs

for h in hooks:
    h.remove()

shape_df = pd.DataFrame(shape_rows)
display(shape_df)

print('final embedding shape =', tuple(emb.shape))
print('first 10 values =', emb[0, :10])


In [ ]:
# Inspect the helper methods around the convolution stack.
with torch.no_grad():
    frame_level_4d = model._get_frame_level_feat(feats)
    frame_level_flat = model.get_frame_level_feat(feats)
    pooled = model.pool(frame_level_4d)
    seg1 = model.seg_1(pooled)

print('input feats [B,T,F]        =', tuple(feats.shape))
print('_get_frame_level_feat      =', tuple(frame_level_4d.shape))
print('get_frame_level_feat       =', tuple(frame_level_flat.shape))
print('pool output               =', tuple(pooled.shape))
print('seg_1 output              =', tuple(seg1.shape))


In [ ]:
# Optional deeper look: shapes of the first block in each residual stage.
block_rows = []
block_hooks = []

for block_name in ['layer1.0', 'layer2.0', 'layer3.0', 'layer4.0']:
    module = dict(model.named_modules())[block_name]
    def _make_hook(name):
        def hook(_module, inputs, outputs):
            block_rows.append({
                'block': name,
                'input_shape': tuple(inputs[0].shape),
                'output_shape': tuple(outputs.shape),
            })
        return hook
    block_hooks.append(module.register_forward_hook(_make_hook(block_name)))

with torch.no_grad():
    _ = model(feats)

for h in block_hooks:
    h.remove()

display(pd.DataFrame(block_rows))
